# CSC4093/DSC4213: Neural Networks and Deep Learning
## Programming Assignment 01 — LSTMs
### Personal Health Mention Classification from Tweets
---
**Task:** Classify tweets as Personal Health Mentions `(1)` or Non-Personal Health Mentions `(0)`  
**Models:** LSTM and Bidirectional LSTM (Bi-LSTM)  
**Embedding:** Keras Embedding Layer (trained end-to-end)  
**Evaluation:** Accuracy, Loss, Confusion Matrix, Classification Report

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout,
    Bidirectional, SpatialDropout1D
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

nltk.download('stopwords')
english_stops = set(stopwords.words('english'))

print('All libraries imported successfully.')

## Step 2: Load and Preview Dataset

In [ ]:
train_data = pd.read_csv('phm_train.csv')
test_data  = pd.read_csv('phm_test.csv')

print('Training set shape:', train_data.shape)
print('Test set shape    :', test_data.shape)
print()
print('Training set preview:')
print(train_data.head())
print()
print('Label distribution — Training set:')
print(train_data['label'].value_counts())
print(f"  Class 0 (Non-Personal): {(train_data['label']==0).sum()}")
print(f"  Class 1 (Personal)    : {(train_data['label']==1).sum()}")
print()
print('Label distribution — Test set:')
print(test_data['label'].value_counts())

## Step 3: Text Preprocessing

In [ ]:
def preprocess_tweets(data):
    x_data = data['tweet'].copy().astype(str)
    y_data = data['label'].copy()

    x_data = x_data.replace({'http\S+|www\S+': ''}, regex=True)   # Remove URLs
    x_data = x_data.replace({'user_mention': ''}, regex=True)      # Remove @mentions placeholder
    x_data = x_data.replace({'<.*?>': ''}, regex=True)             # Remove HTML tags
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex=True)       # Keep only letters
    x_data = x_data.apply(lambda tweet: [
        w.lower() for w in tweet.split()
        if w.lower() not in english_stops and len(w) > 2
    ])
    return x_data, y_data

x_train, y_train = preprocess_tweets(train_data)
x_test,  y_test  = preprocess_tweets(test_data)

print('Sample preprocessed tweet:')
print(x_train.iloc[0])
print()
print('Corresponding label:', y_train.iloc[0])

## Step 4: Tokenization and Padding

In [ ]:
def get_max_length(x_data):
    lengths = [len(t) for t in x_data]
    return int(np.mean(lengths))

# Fit tokenizer ONLY on training data
token = Tokenizer(lower=False)
token.fit_on_texts(x_train)

x_train_seq = token.texts_to_sequences(x_train)
x_test_seq  = token.texts_to_sequences(x_test)

max_length  = get_max_length(x_train)
total_words = len(token.word_index) + 1

x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test_seq,  maxlen=max_length, padding='post', truncating='post')

print('Vocabulary size      :', total_words)
print('Max tweet length     :', max_length)
print('X Train shape        :', x_train_pad.shape)
print('X Test  shape        :', x_test_pad.shape)
print()
print('Encoded X Train (first 3 rows):')
print(x_train_pad[:3])

## Step 5: Compute Class Weights (Handle Imbalance)

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print('Class Weights:')
print(f'  Class 0 (Non-Personal Health): {class_weight_dict[0]:.4f}')
print(f'  Class 1 (Personal Health)    : {class_weight_dict[1]:.4f}')

## Step 6: Hyperparameters

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────
EMBED_DIM      = 128    # Embedding dimensions
LSTM_UNITS     = 128    # Units in LSTM layer
BILSTM_UNITS   = 128    # Units in each direction of Bi-LSTM
SPATIAL_DROP   = 0.2    # SpatialDropout1D after embedding
RECURRENT_DROP = 0.2    # Recurrent dropout inside LSTM cells
DENSE_DROP     = 0.5    # Dropout before final Dense layer
BATCH_SIZE     = 64
EPOCHS         = 15
LR             = 0.001  # Same LR for both — ReduceLROnPlateau handles decay
# ────────────────────────────────────────────────────────────────────
import os
os.makedirs('models', exist_ok=True)
print('Hyperparameters configured.')

## Step 7: Build Model 1 — LSTM

In [ ]:
lstm_model = Sequential(name='LSTM_Model')
lstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
lstm_model.add(SpatialDropout1D(SPATIAL_DROP))                          # Drops entire feature maps — better for NLP than regular Dropout
lstm_model.add(LSTM(LSTM_UNITS, dropout=RECURRENT_DROP, recurrent_dropout=RECURRENT_DROP))
lstm_model.add(Dropout(DENSE_DROP))
lstm_model.add(Dense(64, activation='relu'))
lstm_model.add(Dense(1,  activation='sigmoid'))

lstm_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(lstm_model.summary())

## Step 8: Train LSTM Model

In [ ]:
lstm_callbacks = [
    ModelCheckpoint(
        'models/LSTM_PHM.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

lstm_history = lstm_model.fit(
    x_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=lstm_callbacks,
    verbose=1
)

## Step 9: Build Model 2 — Bi-LSTM

In [ ]:
bilstm_model = Sequential(name='BiLSTM_Model')
bilstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
bilstm_model.add(SpatialDropout1D(SPATIAL_DROP))
bilstm_model.add(Bidirectional(LSTM(BILSTM_UNITS, dropout=RECURRENT_DROP, recurrent_dropout=RECURRENT_DROP)))
bilstm_model.add(Dropout(DENSE_DROP))
bilstm_model.add(Dense(64, activation='relu'))
bilstm_model.add(Dense(1,  activation='sigmoid'))

bilstm_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(bilstm_model.summary())

## Step 10: Train Bi-LSTM Model

In [ ]:
bilstm_callbacks = [
    ModelCheckpoint(
        'models/BiLSTM_PHM.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

bilstm_history = bilstm_model.fit(
    x_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=bilstm_callbacks,
    verbose=1
)

## Step 11: Evaluate Both Models on Test Set

In [ ]:
# LSTM predictions
lstm_probs  = lstm_model.predict(x_test_pad)
lstm_pred   = (lstm_probs >= 0.5).astype(int).flatten()
lstm_correct = np.sum(lstm_pred == np.array(y_test))
lstm_acc     = lstm_correct / len(lstm_pred) * 100

# Bi-LSTM predictions
bilstm_probs  = bilstm_model.predict(x_test_pad)
bilstm_pred   = (bilstm_probs >= 0.5).astype(int).flatten()
bilstm_correct = np.sum(bilstm_pred == np.array(y_test))
bilstm_acc     = bilstm_correct / len(bilstm_pred) * 100

print('=' * 50)
print('LSTM Model — Test Results')
print('=' * 50)
print(f'Correct Predictions : {lstm_correct}')
print(f'Wrong Predictions   : {len(lstm_pred) - lstm_correct}')
print(f'Test Accuracy       : {lstm_acc:.2f}%')
print()
print('Classification Report:')
print(classification_report(
    y_test, lstm_pred,
    target_names=['Non-Personal (0)', 'Personal Health (1)']
))

print('=' * 50)
print('Bi-LSTM Model — Test Results')
print('=' * 50)
print(f'Correct Predictions : {bilstm_correct}')
print(f'Wrong Predictions   : {len(bilstm_pred) - bilstm_correct}')
print(f'Test Accuracy       : {bilstm_acc:.2f}%')
print()
print('Classification Report:')
print(classification_report(
    y_test, bilstm_pred,
    target_names=['Non-Personal (0)', 'Personal Health (1)']
))

## Step 12: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices — LSTM vs Bi-LSTM', fontsize=14, fontweight='bold')

labels = ['Non-Personal (0)', 'Personal Health (1)']

# LSTM Confusion Matrix
cm_lstm = confusion_matrix(y_test, lstm_pred)
sns.heatmap(
    cm_lstm, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels, ax=axes[0],
    linewidths=0.5, linecolor='gray'
)
axes[0].set_title(f'LSTM  (Accuracy: {lstm_acc:.2f}%)', fontsize=12)
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')
axes[0].tick_params(axis='x', rotation=15)

# Bi-LSTM Confusion Matrix
cm_bilstm = confusion_matrix(y_test, bilstm_pred)
sns.heatmap(
    cm_bilstm, annot=True, fmt='d', cmap='Greens',
    xticklabels=labels, yticklabels=labels, ax=axes[1],
    linewidths=0.5, linecolor='gray'
)
axes[1].set_title(f'Bi-LSTM  (Accuracy: {bilstm_acc:.2f}%)', fontsize=12)
axes[1].set_ylabel('Actual Label')
axes[1].set_xlabel('Predicted Label')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

## Step 13: Accuracy and Loss Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LSTM vs Bi-LSTM — Training Performance', fontsize=15, fontweight='bold')

# LSTM Accuracy
axes[0,0].plot(lstm_history.history['accuracy'],     label='Train', color='royalblue',  linewidth=2)
axes[0,0].plot(lstm_history.history['val_accuracy'], label='Val',   color='darkorange', linewidth=2)
axes[0,0].set_title('LSTM — Accuracy')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# LSTM Loss
axes[0,1].plot(lstm_history.history['loss'],     label='Train', color='royalblue',  linewidth=2)
axes[0,1].plot(lstm_history.history['val_loss'], label='Val',   color='darkorange', linewidth=2)
axes[0,1].set_title('LSTM — Loss')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Loss')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# Bi-LSTM Accuracy
axes[1,0].plot(bilstm_history.history['accuracy'],     label='Train', color='seagreen', linewidth=2)
axes[1,0].plot(bilstm_history.history['val_accuracy'], label='Val',   color='crimson',  linewidth=2)
axes[1,0].set_title('Bi-LSTM — Accuracy')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Accuracy')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# Bi-LSTM Loss
axes[1,1].plot(bilstm_history.history['loss'],     label='Train', color='seagreen', linewidth=2)
axes[1,1].plot(bilstm_history.history['val_loss'], label='Val',   color='crimson',  linewidth=2)
axes[1,1].set_title('Bi-LSTM — Loss')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Loss')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_loss_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: accuracy_loss_plots.png')

## Step 14: Performance Comparison Table

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

lstm_precision = precision_score(y_test, lstm_pred,   average='weighted') * 100
lstm_recall    = recall_score(y_test,    lstm_pred,   average='weighted') * 100
lstm_f1        = f1_score(y_test,        lstm_pred,   average='weighted') * 100

bi_precision   = precision_score(y_test, bilstm_pred, average='weighted') * 100
bi_recall      = recall_score(y_test,    bilstm_pred, average='weighted') * 100
bi_f1          = f1_score(y_test,        bilstm_pred, average='weighted') * 100

comparison = pd.DataFrame({
    'Metric'    : ['Test Accuracy (%)', 'Precision (%)', 'Recall (%)',
                   'F1-Score (%)', 'Best Val Accuracy (%)', 'Best Val Loss'],
    'LSTM'      : [
        f'{lstm_acc:.2f}',
        f'{lstm_precision:.2f}',
        f'{lstm_recall:.2f}',
        f'{lstm_f1:.2f}',
        f"{max(lstm_history.history['val_accuracy'])*100:.2f}",
        f"{min(lstm_history.history['val_loss']):.4f}"
    ],
    'Bi-LSTM'   : [
        f'{bilstm_acc:.2f}',
        f'{bi_precision:.2f}',
        f'{bi_recall:.2f}',
        f'{bi_f1:.2f}',
        f"{max(bilstm_history.history['val_accuracy'])*100:.2f}",
        f"{min(bilstm_history.history['val_loss']):.4f}"
    ]
})

print('=' * 55)
print('        Model Performance Comparison')
print('=' * 55)
print(comparison.to_string(index=False))
print('=' * 55)

## Step 15: Discussion

### Model Architectures

Both models follow the same single-layer recurrent architecture for a fair, controlled comparison:

```
Embedding (128 dim)
    ↓
SpatialDropout1D (0.2)
    ↓
LSTM / Bi-LSTM (128 units, recurrent_dropout=0.2)
    ↓
Dropout (0.5)
    ↓
Dense (64, ReLU)
    ↓
Dense (1, Sigmoid)
```

### Key Design Decisions

| Technique | Reason |
|---|---|
| **SpatialDropout1D** | Drops entire embedding feature maps — more effective than regular Dropout for sequential NLP data |
| **Recurrent Dropout** | Applied inside LSTM cells to regularize hidden state transitions |
| **Class Weight Balancing** | Compensates for unequal distribution of personal vs non-personal tweets |
| **ReduceLROnPlateau** | Halves learning rate when val_loss stagnates — prevents getting stuck |
| **EarlyStopping (patience=4)** | Stops training when val_accuracy stops improving, restores best weights |
| **ModelCheckpoint** | Saves only the best model based on val_accuracy |

### LSTM Model
The standard LSTM processes tweets in a **single forward direction**, learning sequential patterns from left to right. It is computationally efficient and well-suited for short tweet sequences where most contextual signals flow naturally in a forward direction.

### Bi-LSTM Model
The Bi-LSTM processes each tweet in **both forward and backward directions simultaneously**, then concatenates both hidden states. This gives the model a complete view of the tweet at every timestep — especially useful when a health-related keyword near the end of the tweet changes the meaning of earlier words.

### Confusion Matrix Interpretation
- **True Positives (TP):** Personal health tweets correctly identified
- **True Negatives (TN):** Non-personal tweets correctly identified
- **False Positives (FP):** Non-personal tweets misclassified as personal (Type I error)
- **False Negatives (FN):** Personal health tweets missed by the model (Type II error)

A lower FN rate is particularly important in health mention detection, as missing a genuine personal health mention is more costly than a false alarm.